# Python: HTTP Clients and APIs
## Introduction to HTTP and the Requests Library

HTTP (HyperText Transfer Protocol) is the foundation of data communication on the web. When your Python code talks to a web API, it does so by sending HTTP requests and receiving HTTP responses. Each request uses a **method** (GET, POST, PUT, DELETE, etc.) to indicate the intended action, and each response carries a **status code** (e.g. 200 OK, 404 Not Found, 500 Internal Server Error) along with a response body.

Python's built-in `urllib` module can handle HTTP, but the third-party [`requests`](https://docs.python-requests.org/) library is almost universally preferred for its clean, human-friendly API:

```python
pip install requests
```

With `requests` you can:
- Send GET, POST, PUT, DELETE and other HTTP requests in one line
- Pass query parameters, headers, and JSON bodies with simple keyword arguments
- Automatically decode JSON responses via `.json()`
- Handle sessions, cookies, authentication, and timeouts with minimal boilerplate

The examples in this notebook use the public [JSONPlaceholder](https://jsonplaceholder.typicode.com/) API — a free, read-only REST API that returns fake data, perfect for practising HTTP calls without needing credentials.

## Performing GET Requests

A **GET request** is the most common HTTP method. It asks the server to retrieve and return a resource — your client sends a request to a URL, and the server responds with data (typically HTML or JSON).

Key characteristics of HTTP GET requests:
- **Retrieving data from the server** — GET is read-only; it never modifies server state
- **Idempotent** — making the same GET request multiple times always produces the same result
- **Limited data size** — parameters are passed in the URL, so there is a practical length limit
- **Can be cached and bookmarked** — because GET requests are predictable and side-effect-free, browsers and proxies can cache them

With the `requests` library, a GET request is a single function call:

```python
import requests

response = requests.get("http://127.0.0.1/api/items")
print(response.status_code)  # e.g. 200
print(response.json())       # parsed response body
```

### Performing GET Requests | Demo 1: Making a Basic GET Request

In [1]:
import requests

response = requests.get("http://127.0.0.1:8000/api/items")
print(response)

<Response [200]>


### Performing GET Requests | Demo 2: Inspecting Response Data

In [2]:
import requests

response = requests.get("http://127.0.0.1:8000/something")
print(response.status_code)

# HTTP Status Codes
if response.status_code == 200: # Success!
    print("Success!")
elif response.status_code == 500: # Server error.
    print("Server error.")
elif response.status_code == 404: # Page not Found.
    print("Page not Found.")

response = requests.get("http://127.0.0.1:8000/api/items")

# Binary data but formatted by the print function
# print(response.content)

# Hexadecimal representation of the binary data
# print(response.content.hex())

# String representation of the response
# print(response.text)

# Response headers contain the metadata for the response
# print(response.headers["content-type"])

# JSON from the response converted to Python objects
print(response.json())
print(response.json()[1]["name"])

404
Page not Found.
[{'name': 'Foo', 'price': 23.45}, {'name': 'Bar', 'price': 67.89}, {'name': 'Baz', 'price': 12.34}, {'name': 'Qux', 'price': 56.78}, {'name': 'Quux', 'price': 45.67}, {'name': 'Corge', 'price': 78.9}, {'name': 'Grault', 'price': 90.12}, {'name': 'Garply', 'price': 34.56}, {'name': 'Waldo', 'price': 89.01}, {'name': 'Fred', 'price': 67.23}, {'name': 'Plugh', 'price': 45.89}, {'name': 'Xyzzy', 'price': 23.78}, {'name': 'Thud', 'price': 90.23}]
Bar


### Performing GET Requests | Demo 3: Passing Query String Parameters

In [3]:
import requests

# Hardcoded query parameters in the URL string
# response = requests.get(
#     "http://127.0.0.1:8000/api/items?offset=2&limit=2&max_price=40"
# )

query_params = {
    "offset": 2,
    "limit": 2,
    "max_price": 40
}

# Use the params keyword argument to pass the query parameters
response = requests.get(
    "http://127.0.0.1:8000/api/items",
    params=query_params,
)

print(response.json())

[{'name': 'Garply', 'price': 34.56}, {'name': 'Xyzzy', 'price': 23.78}]


## Sending Data with POST Requests

While GET retrieves data, a **POST request** sends data to the server. The data travels in the **request body** rather than the URL, which allows for larger payloads and keeps sensitive data out of browser history and server logs.

Key characteristics of HTTP POST requests:
- **Submitting data to a server** — used for form submissions, file uploads, and creating new resources
- **Sending data in the request body** — unlike GET, data is not exposed in the URL
- **Specify the content type in the headers** — the `Content-Type` header tells the server how to interpret the body (e.g. `application/x-www-form-urlencoded`, `application/json`, `multipart/form-data`)
- **More secure than GET requests** — body data is not stored in browser history, server logs, or bookmarks

With `requests`, you can POST form data, JSON, or files using different keyword arguments:

```python
import requests

# Form data (Content-Type: application/x-www-form-urlencoded)
requests.post(url, data={"key": "value"})

# JSON data (Content-Type: application/json)
requests.post(url, json={"key": "value"})

# File upload (Content-Type: multipart/form-data)
requests.post(url, files={"file": open("file.csv", "rb")})
```

After a successful POST, servers often respond with a `303 See Other` redirect, sending the client to a new URL (e.g. the newly created resource page).

### Sending Data with POST Requests | Demo 1: Submitting Form Data with POST

In [4]:
import requests

# Submit form data through the body of the request
response = requests.post(
    "http://127.0.0.1:8000/items/new",
    data={"name": "Another item", "price": 44},
    allow_redirects=False, # We don't want to get redirected to the new item page
)

# Request content type is x-www-form-urlencoded and the body is the form data
# You can use response.request to access the prepared request
print(response.request.headers["content-type"])
print(response.request.body)

application/x-www-form-urlencoded
name=Another+item&price=44


### Sending Data with POST Requests | Demo 2: Sending JSON Data to APIs

In [5]:
import requests

# Message body to be sent in the request
message_body = {"name": "Some item", "price": 22}

# Submit JSON data through the body of the request
response = requests.post(
    "http://127.0.0.1:8000/api/items",
    json=message_body # use json keyword argument to send JSON data
)

# The content type is application/json and the body is the JSON data
print(response.request.headers["content-type"])
print(response.request.body)

application/json
b'{"name": "Some item", "price": 22}'


### Sending Data with POST Requests | Demo 3: Uploading Files

In [6]:
import requests
from pathlib import Path

# Resolve file paths relative to this notebook's location
assets = Path(".") / "assets"

# Manage multiple files using context manager
with open(assets / "file1.csv", "rb") as file1, open(assets / "file2.csv", "rb") as file2:
    # Create a list of files
    files = [
        ("files", ("file1.csv", file1, "text/csv")),
        ("files", ("file2.csv", file2, "text/csv")),
    ]

    response = requests.post(
        "http://127.0.0.1:8000/upload-files",
        files=files, # use files keyword argument to send files
    )

print(response.json())

{'uploaded_files': ['file1.csv', 'file2.csv']}


### Sending Data with POST Requests | Demo 4: Using Other HTTP Methods

In [7]:
import requests

# PUT request typically replaces the entire resource with new data
response = requests.put(
    "http://127.0.0.1:8000/api/items/1",
    json={"name": "Updated PUT Name", "price": 100}
)

# PATCH request typically updates specific fields of a resource
response = requests.patch(
    "http://127.0.0.1:8000/api/items/1",
    json={"name": "Updated PATCH Name"}
)

# DELETE request typically deletes a resource
response = requests.delete(
    "http://127.0.0.1:8000/api/items/1",
)

print(response.json())

{'status': 'Item deleted', 'item': {'name': 'Updated PATCH Name', 'price': 100.0}}


## A Note on HTTP Methods: Browsers vs. HTTP Clients

You might have heard that "browsers only understand GET and POST." This is **partially true** — but the distinction matters.

### The HTML Form Limitation

HTML `<form>` elements natively support only two methods:

```html
<form method="GET" ...>   <!-- fetches data -->
<form method="POST" ...>  <!-- submits data -->
```

Any other value is ignored and the browser falls back to GET. This is a limitation of the **HTML specification**, not of browsers themselves.

To work around this in traditional web applications, frameworks like Rails, Laravel, and Django use a hidden `_method` field — you submit a POST but tell the server to treat it as a DELETE or PUT:

```html
<form method="POST">
    <input type="hidden" name="_method" value="DELETE">
    ...
</form>
```

The server reads `_method` and routes accordingly. In this sense, PUT/PATCH/DELETE were "faked on top of POST" — which is likely where the idea of them being "decorators" comes from.

### Modern Browsers Do Understand All Methods

Browsers themselves are not limited to GET and POST. JavaScript running in the browser can send any HTTP method freely using `fetch` or `XMLHttpRequest`:

```javascript
// JavaScript in a browser — works fine
fetch("/api/items/1", { method: "DELETE" });
fetch("/api/items/1", { method: "PUT", body: JSON.stringify({name: "New"}) });
```

### Python's `requests` Has No Such Limitation

When you use Python's `requests` library, you are acting as a pure HTTP client — there is no HTML form involved at all. PUT, PATCH, and DELETE are fully first-class methods:

```python
requests.put(url, json={...})     # replaces the full resource
requests.patch(url, json={...})   # updates specific fields
requests.delete(url)              # removes the resource
```

These are not workarounds or decorators — they send the real HTTP method directly over the wire, and a well-designed REST API will handle each one differently by convention:

| Method | Typical use |
|--------|-------------|
| `GET` | Read a resource |
| `POST` | Create a new resource |
| `PUT` | Replace an entire resource |
| `PATCH` | Update specific fields of a resource |
| `DELETE` | Remove a resource |

## Handling Different Response Formats

HTTP responses can come back in different formats depending on the API. The three most common are **JSON**, **XML**, and **HTML**. Python's `requests` library gives you the raw response body as bytes or text — it's up to you to parse it into something useful.

### JSON

JSON is the dominant format for modern REST APIs. `requests` handles it natively via `.json()`, which deserializes the response body into Python objects automatically.

When `.json()` parses a response, JSON types are converted to their Python equivalents:

| JSON | Python |
|------|--------|
| Object | `dict` |
| Array | `list` |
| String | `str` |
| Number | `int` / `float` |
| `null` | `None` |

Always wrap `.json()` in a `try/except ValueError` — if the server returns non-JSON (e.g. an HTML error page), it will raise an exception.

### XML

Older APIs and some enterprise systems still return XML. Python's standard library includes `xml.etree.ElementTree` for parsing it. You send the XML as a plain string in the request body with `Content-Type: application/xml`, then parse the response text with `ET.fromstring()`.

### HTML

Sometimes you need to scrape data from an HTML response (e.g. a server that renders pages rather than returning JSON). The third-party `beautifulsoup4` library makes this straightforward — parse the response text with `BeautifulSoup`, then use `.find()` and `.get_text()` to extract specific elements by tag, class, or ID.

### Handling Different Response Formats | Demo 1: Working with JSON Responses

In [8]:
import requests

response = requests.get("http://127.0.0.1:8000/api/items")

# Try to parse the response as JSON
try:
    data = response.json()
except ValueError:
    print("Response is not valid JSON")

# Pretty print the data
import json

print(json.dumps(data, indent=4))

[
    {
        "name": "Foo",
        "price": 23.45
    },
    {
        "name": "Baz",
        "price": 12.34
    },
    {
        "name": "Qux",
        "price": 56.78
    },
    {
        "name": "Quux",
        "price": 45.67
    },
    {
        "name": "Corge",
        "price": 78.9
    },
    {
        "name": "Grault",
        "price": 90.12
    },
    {
        "name": "Garply",
        "price": 34.56
    },
    {
        "name": "Waldo",
        "price": 89.01
    },
    {
        "name": "Fred",
        "price": 67.23
    },
    {
        "name": "Plugh",
        "price": 45.89
    },
    {
        "name": "Xyzzy",
        "price": 23.78
    },
    {
        "name": "Thud",
        "price": 90.23
    },
    {
        "name": "Another item",
        "price": 44.0
    },
    {
        "name": "Some item",
        "price": 22.0
    }
]


### Handling Different Response Formats | Demo 2: Parsing XML Responses

In [9]:
import requests
# Use the xml library to parse the response
import xml.etree.ElementTree as ET

# XML message body (which is just a string)
message_body = """
<item>
    <name>Some item</name>
    <price>300</price>
</item>
"""

response = requests.post(
    "http://127.0.0.1:8000/api/items/xml",
    data=message_body,
    headers={"Content-Type": "application/xml"} # Set the request content type to xml
)

# Print the raw response
print(response.text)

# Parse the response as XML and find the name and price
print(ET.fromstring(response.text).find("name").text)
print(ET.fromstring(response.text).find("price").text)


        <response>
            <name>Some item</name>
            <price>300</price>
        </response>
        
Some item
300


### Handling Different Response Formats | Demo 3: Extracting Data from HTML

In [10]:
import requests
from bs4 import BeautifulSoup

# Make a request to the /about route
response = requests.get("http://127.0.0.1:8000/about")

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Extract the description (paragraph text)
description = soup.find('p').get_text().strip() # use strip to remove whitespace
print("Description:")
print(description)

# Extract phone and email by using IDs
phone = soup.find('td', id='phone').get_text().strip()
email = soup.find('td', id='email').get_text().strip()

print("Phone:", phone)
print("Email:", email)

Description:
Welcome to our premium items webshop! We are a leading online retailer.
Phone: +1 (555) 123-4567
Email: info@itemsshop.com


## Sessions, Cookies, and Authentication

HTTP is stateless — each request is independent and the server has no memory of previous requests. **Cookies** and **sessions** are the mechanisms used to maintain state across multiple requests.

### Cookies

When you log in to a server, it can respond with a `Set-Cookie` header containing a small piece of data (e.g. `user-id=RWJjMTIz`). Your client stores this cookie and sends it automatically in the `Cookie` header on every subsequent request. The server reads the cookie to identify you without requiring you to log in again.

With `requests`, you can pass cookies manually using the `cookies` argument, or let a `Session` manage them automatically.

### Sessions

A `requests.Session` object persists cookies, headers, and other state across multiple requests to the same host. It is the cleanest way to handle authenticated workflows — you log in once and the session automatically attaches the returned cookie to every following request.

You can also configure session-level defaults:
```python
session = requests.Session()
session.timeout = 5          # default timeout for all requests
session.max_redirects = 3    # max redirects to follow
```

### Authentication Methods

Beyond cookie-based auth, `requests` supports several authentication strategies:

- **HTTP Basic Auth** — sends `username:password` base64-encoded in the `Authorization` header. Use `auth=HTTPBasicAuth(user, pass)` or just `auth=(user, pass)`.
- **JWT / Bearer tokens** — send a token in `Authorization: Bearer <token>`. Implement a custom `AuthBase` subclass to attach it automatically.

### HTTPS and SSL/TLS Certificates

When you use an `https://` URL, `requests` verifies the server's SSL/TLS certificate against trusted Certificate Authorities (CAs) by default. If verification fails, it raises an `SSLError`.

```python
# Default — verifies the certificate
requests.get("https://api.example.com/data")

# Disable verification (not recommended for production)
requests.get("https://api.example.com/data", verify=False)

# Provide a custom CA bundle
requests.get("https://api.example.com/data", verify="/path/to/bundle.crt")

# Provide a client certificate (for mutual TLS)
requests.get("https://api.example.com/data", cert=("/cert_path", "/key_path"))
```

### Sessions, Cookies, and Authentication | Demo 1: Maintaining State with Cookies

In [11]:
import requests

# Set a custom cookie
custom_cookies = {"user_id": "2"}

response = requests.get(
    "http://127.0.0.1:8000/api/cookies",
    cookies=custom_cookies # use the cookies argument
)

# Print the cookiejar from the response as a dictionary
print(response.cookies.get_dict())
# Print the value of the user_id cookie
print(response.cookies["user_id"])

# Proof that cookies are managed with headers
print("=== Request Headers ===")
print(response.request.headers)

print("\n=== Response Headers ===")
print(response.headers)

{'user_id': '2'}
2
=== Request Headers ===
{'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive', 'Cookie': 'user_id=2'}

=== Response Headers ===
{'date': 'Tue, 22 Sep 2026 13:25:11 GMT', 'server': 'uvicorn', 'content-length': '16', 'content-type': 'application/json', 'set-cookie': 'user_id=2; Path=/; SameSite=lax'}


### Sessions, Cookies, and Authentication | Demo 2: Implementing Authentication with Sessions

#### 2a. Authenticating with Cookies

In [12]:
import requests

# Wrong credentials
# credentials = {"username": "name", "password": "pass"}
# Correct credentials
credentials = {"username": "some_name", "password": "pass"}

# Log in with the credentials and receive cookies
login_response = requests.post(
    "http://127.0.0.1:8000/api/login",
    data=credentials
)

# Extract cookies returned from login response
login_cookies = login_response.cookies

print("Cookies returned from login:")
print(login_cookies.get_dict())

print("Login response:")
print(login_response.text)

# Use the cookies to access a protected route
response = requests.get(
    "http://127.0.0.1:8000/protected",
    cookies=login_cookies
)

print("Protected route:")
print(response.status_code)
print(response.text)

Cookies returned from login:
{'user_id': 'b36aa914efd60376dc0d4a2c7cadd481'}
Login response:
{"message":"Login successful"}
Protected route:
200
{"message":"You have access to this protected route"}


#### 2b. Authenticating with Session

In [13]:
import requests

# session = requests.Session() # use the session object to store the cookies

# Use the context manager to manage the session
with requests.Session() as session:
    credentials = {"username": "some_name", "password": "pass"}

    login_response = session.post("http://127.0.0.1:8000/api/login", data=credentials)

    print("Cookies returned from login:")
    print(session.cookies.get_dict())
    print("Login response:")
    print(login_response.text)

    response = session.get("http://127.0.0.1:8000/protected")

    print("Protected route:")
    print(response.status_code)
    print(response.text)

Cookies returned from login:
{'user_id': '40db4fd130a5dcb25b9883981aad0e24'}
Login response:
{"message":"Login successful"}
Protected route:
200
{"message":"You have access to this protected route"}


#### 2c. Session Login (Concise)

In [14]:
import requests

with requests.Session() as session:
    credentials = {"username": "some_name", "password": "pass"}

    session.post("http://127.0.0.1:8000/api/login", data=credentials)

    response = session.get("http://127.0.0.1:8000/protected")

    print("Protected route:")
    print(response.status_code)
    print(response.text)

Protected route:
200
{"message":"You have access to this protected route"}


### Sessions, Cookies, and Authentication | Demo 3: Exploring Other Authentication Methods

#### 3a. Authenticating with HTTPBasicAuth

In [15]:
import requests
from requests.auth import HTTPBasicAuth

username = "username"
password = "pass"

response = requests.get(
    "http://127.0.0.1:8000/protected-endpoint",
    auth=HTTPBasicAuth(username, password)
) # you can leave out HTTPBasicAuth, it will be used by default

print(response.text)

{"message":"Welcome, authenticated user!"}


#### 3b. Authenticating with JSON Web Tokens

In [16]:
import requests
from requests.auth import AuthBase

# Authorization: Bearer <token>

class JWTAuth(AuthBase):
    def __init__(self, token):
        self.token = token

    # The auth class must have a __call__ method that takes a request object and returns a request object
    def __call__(self, request):
        request.headers["Authorization"] = f"Bearer {self.token}"
        return request

token = "abcde123"
response = requests.get(
    "http://127.0.0.1:8000/jwt-protected-route",
    auth=JWTAuth(token)
)

print(response.text)

{"message":"Access to protected route granted"}


### Sessions, Cookies, and Authentication | Demo 4: Securing Communication with TLS

In [17]:
# Examples of how to use TLS certificates for HTTPS protected communication

import requests
import urllib3

# Using TLS by default. If the verification failed, it will raise SSLError
response = requests.get("https://www.weather.com/my-town")

# Disable verification if needed (not recommended for production)
# Suppress the InsecureRequestWarning that verify=False produces
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
response = requests.get("https://www.weather.com/my-town", verify=False)

# Provide a custom CA bundle (replace with a real path in production)
# response = requests.get("https://www.weather.com/my-town", verify="/path/to/ca-bundle.crt")

# Some servers require client certificates (replace with real cert/key paths)
# response = requests.get("https://www.weather.com/my-town", cert=("/path/to/cert.pem", "/path/to/key.pem"))

print(response.status_code)

404


## Error Handling and Resilient Clients

Real-world HTTP clients need to handle failures gracefully. Servers can return error status codes, connections can time out, responses can redirect, and flaky services may require retries. Building a resilient client means anticipating these scenarios and handling them explicitly.

### Handling HTTP Errors

The `requests` library does **not** raise exceptions for HTTP error status codes (4xx, 5xx) by default — it simply returns the response. Call `response.raise_for_status()` to explicitly raise an `HTTPError` if the status code indicates a failure.

```python
response = requests.get(url)
response.raise_for_status()  # raises HTTPError for 4xx/5xx
```

### Timeouts

By default, `requests` will wait indefinitely for a server to respond. Always set a timeout to prevent hanging requests. The timeout can be a single number (both connect and read) or a tuple `(connect_timeout, read_timeout)`:

- **Connect timeout** — how long to wait to establish a TCP connection
- **Read timeout** — how long to wait for the server to send data after connecting

### Redirects and History

By default, `requests` follows redirects automatically. The full chain of intermediate responses is stored in `response.history`. You can inspect it, limit `max_redirects`, or disable following entirely with `allow_redirects=False`.

To set a session-level redirect limit:
```python
session = requests.Session()
session.max_redirects = 3
```

To iterate through the redirect chain:
```python
for resp in response.history:
    print(resp.status_code, resp.url)
```

### Retry Logic

Transient failures (network blips, 500 errors) can be handled with automatic retries using `urllib3`'s `Retry` object mounted on a `Session`:

```python
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

retries = Retry(total=3, backoff_factor=0.1, status_forcelist=[500])
session.mount("http://", HTTPAdapter(max_retries=retries))
```

The `backoff_factor` introduces an increasing delay between retries to avoid hammering a struggling server.

### Error Handling and Resilient Clients | Demo 1: Handling HTTP Errors

In [18]:
import requests

try:
    # Make a request to the non-existent route
    response = requests.get("http://127.0.0.1:8000/something/")
    response.raise_for_status()
except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
except Exception as err:
    print(err)
else:
    print(response.status_code)

HTTP error occurred: 404 Client Error: Not Found for url: http://127.0.0.1:8000/something/


### Error Handling and Resilient Clients | Demo 2: Using Timeouts to Prevent Hanging Requests

In [19]:
import requests

try:
    # The timeout is a tuple of (connect timeout, read timeout)
    # The connect timeout is the time to connect to the server
    # The read timeout is the time to read the response from the server
    response = requests.get("http://127.0.0.1:8000/slow-response", timeout=(5, 3))
    print(response.json())

    # This will raise a ConnectTimeout error
    # response = requests.get("http://10.255.255.1", timeout=(5, 6))
except requests.exceptions.ConnectTimeout:
    print("The request failed to connect in the allotted time.")
except requests.exceptions.ReadTimeout:
    print("The server did not send any data in the allotted amount of time.")
except requests.exceptions.Timeout:
    print("A timeout error occurred.")

The server did not send any data in the allotted amount of time.


### Error Handling and Resilient Clients | Demo 3: Managing Redirection and History

In [20]:
import requests

# This will redirect to the new route
# response = requests.get("http://127.0.0.1:8000/old-route")

# HEAD HTTP method does not allow redirects by default
# This will return a 307 redirect status code
# response = requests.head("http://127.0.0.1:8000/old-route")

# You can set the allow_redirects parameter to True to allow redirects
# response = requests.head("http://127.0.0.1:8000/old-route", allow_redirects=True)

# You can also disable redirection for the GET request
# response = requests.get("http://127.0.0.1:8000/old-route", allow_redirects=False)

response = requests.get("http://127.0.0.1:8000/old-route")

# Print the history of redirects
print(response.history)

print(response.url)
print(response.status_code)
print(response.text)

[<Response [307]>]
http://127.0.0.1:8000/new-route
200
{"message":"This is the new route!"}


### Error Handling and Resilient Clients | Demo 4: Implementing Retry Logic

In [21]:
import logging
import requests
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

# Set the basic config for the logging
logging.basicConfig(level=logging.DEBUG)
requests_log = logging.getLogger("urllib3")
requests_log.setLevel(logging.DEBUG)
requests_log.propagate = True

# Create a session object
session = requests.Session()

# Set the retries for the session
# total: total number of retries
# backoff_factor: factor to multiply the backoff time
# status_forcelist: list of status codes to force a retry
# allowed_methods: list of methods to allow retries for
retries = Retry(total=3, backoff_factor=0.1, status_forcelist=[500], allowed_methods={"GET"})
session.mount("http://127.0.0.1", HTTPAdapter(max_retries=retries))

# Or you can just use an integer instead of the Retry object to set the total number of retries
# session.mount("http://127.0.0.1", HTTPAdapter(max_retries=3))

try:
    # Flaky endpoint is a route that randomly fails or succeeds
    response = session.get("http://127.0.0.1:8000/flaky")
    print("Final response status:", response.status_code)
except RetryError:
    print("Maximum retries exceeded. Server is not available.")

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): 127.0.0.1:8000
DEBUG:urllib3.connectionpool:http://127.0.0.1:8000 "GET /flaky HTTP/1.1" 500 25
DEBUG:urllib3.util.retry:Incremented Retry for (url='/flaky'): Retry(total=2, connect=None, read=None, redirect=None, status=None)
DEBUG:urllib3.connectionpool:Retry: /flaky
DEBUG:urllib3.connectionpool:http://127.0.0.1:8000 "GET /flaky HTTP/1.1" 500 25
DEBUG:urllib3.util.retry:Incremented Retry for (url='/flaky'): Retry(total=1, connect=None, read=None, redirect=None, status=None)
DEBUG:urllib3.connectionpool:Retry: /flaky
DEBUG:urllib3.connectionpool:http://127.0.0.1:8000 "GET /flaky HTTP/1.1" 200 21


Final response status: 200
